In [311]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
import os
import operator

In [312]:
load_dotenv()

True

In [313]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')

In [314]:
class diagnosisSchema(BaseModel):
    issue_type: Literal["UX", "performance", "Bug", "Support", "Other"] = Field(description='What is the issue mentioned in the review')
    tone: Literal["Frustrated","disappointed","calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low","high","medium"] = Field(description='How urgent is the issue is')

In [315]:
structured_model = llm.with_structured_output(SentimentSchema)
structured_model2 = llm.with_structured_output(diagnosisSchema)


In [316]:
class sentiment(TypedDict):
    sentence: str
    sentimentvalue: Literal["positive","negative"]
    diagnosis: dict
    response: str
    response: str


In [317]:
api_key = os.getenv("GROQ_API")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing from .env")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=api_key
)

In [318]:
def find_sentiment(state: sentiment):
    prompt = f"""
    Classify the sentiment of this customer feedback
    as either positive or negative.

    Feedback: {state['sentence']}
    """

    result = structured_model.invoke(prompt).sentiment

    return {'sentimentvalue': result}

In [319]:
def condition(state: sentiment) -> Literal["negative","positive"]:
    if state["sentimentvalue"] == 'positive':
        return "positive_response"
    else:
        return "run_diagnosis"

In [320]:
def run_diagnosis(state: sentiment):
    prompt = f"""
    Analyze the following customer feedback.

    Return these three things:
    1. Issue type: Why is the response negative? What is the problem?
    2. Tone: What is the tone of the sentence?
    3. Urgency: How urgent is the issue?

    Sentence: {state['sentence']}
    """

    result = structured_model2.invoke(prompt)

    return {'diagnosis': result}

In [321]:
def negative_response(state: sentiment):
    prompt =  f'give the negative response of the sentence below /n {state["sentence"]}'
    result= structured_model.invoke(prompt)
    return {'negative_response' : result}

def positive_response(state: sentiment):
    prompt= f'give the positive response of the sentence below /n {state["sentence"]}'
    result= structured_model.invoke(prompt)
    return {' positive_response' : result}

In [322]:
graph = StateGraph(sentiment)
graph.add_node('find_sentiment',find_sentiment)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)
graph.add_node('positive_response',positive_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges(
    'find_sentiment',
    condition,
    {
        'run_diagnosis': 'run_diagnosis',
        'positive_response': 'positive_response',
    })
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response',END)
graph.add_edge('positive_response',END)



In [324]:

workflow = graph.compile()
workflow.invoke({
    'sentence':'The product quality was terrible, and the customer service was extremely disappointing.'
})

{'sentence': 'The product quality was terrible, and the customer service was extremely disappointing.',
 'sentimentvalue': 'negative',
 'diagnosis': diagnosisSchema(issue_type='Bug', tone='Frustrated', urgency='high')}